# 第19课：RAG - 检索增强生成

## 学习目标
- 理解 RAG 的核心动机：为什么大模型需要外挂知识库
- 掌握 RAG 的三段式流程：检索 -> 增强 -> 生成
- 理解向量检索、Embedding、Chunking 等关键工程概念
- 用代码实现一个最小 RAG 系统

## 核心概念：为什么需要 RAG？

大模型（如 GPT、BERT）有两个根本性局限：
1. **知识截止**：训练数据有时间边界，不知道最新信息
2. **幻觉问题**：不确定时容易编造听起来合理但错误的内容

RAG（Retrieval-Augmented Generation）的思路很简单——**让模型在回答之前先查资料**：
- 就像开卷考试：不确定的时候翻书，而不是凭记忆猜
- 把检索外部知识和生成回答结合起来

**在 AI 演进中的位置**：
- 上一步：大模型微调（让模型学会特定领域的表达方式）
- 本课：RAG（让模型能动态访问外部知识，不依赖重训练）
- 下一步：Agent（让模型自主决定何时检索、调用什么工具）

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Step 1: 文档分块（Chunking）

真实 RAG 系统中，知识库可能是几十万页的文档。我们不能把整篇文档丢给模型，需要先切成小块。

**为什么？**
- 模型有上下文窗口限制
- 小块检索更精准（相关性更高）
- 常见策略：固定长度切分、按段落切分、递归切分

In [ ]:
# 模拟一个小型知识库
documents = [
    "Python 是一种解释型高级编程语言，由 Guido van Rossum 于 1991 年发布。Python 强调代码可读性。",
    "机器学习是人工智能的一个子领域，通过数据训练模型，使计算机能够从经验中学习。",
    "Transformer 架构于 2017 年在论文 Attention is All You Need 中提出，彻底改变了 NLP 领域。",
    "RAG（检索增强生成）由 Meta 在 2020 年提出，将信息检索与文本生成结合，减少大模型的幻觉问题。",
    "向量数据库（如 FAISS、Milvus）用于存储和高效检索高维向量，是 RAG 系统的核心基础设施。",
    "LangChain 是一个流行的 LLM 应用开发框架，提供了 RAG、Agent、Chain 等抽象。",
    "Embedding 是将文本映射到高维向量空间的技术，语义相近的文本在向量空间中距离更近。",
    "大模型微调（Fine-tuning）是在预训练模型基础上，用领域数据进一步训练，使模型适应特定任务。"
]

print(f"知识库共 {len(documents)} 个文档块")
for i, doc in enumerate(documents):
    print(f"  [{i}] {doc[:50]}...")

## Step 2: 向量化与索引（Embedding + Indexing）

RAG 的核心步骤：把文本转成向量，然后用向量相似度做检索。

这里用 TF-IDF 做简化演示（生产系统用 Sentence-BERT、OpenAI Embedding 等）：

In [ ]:
# 用 TF-IDF 将文档向量化
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

print(f"文档向量矩阵形状: {doc_vectors.shape}")
print(f"（{doc_vectors.shape[0]} 个文档 x {doc_vectors.shape[1]} 个特征词）")

# 可视化：展示 query 与各文档的相似度
query = "什么是 RAG？"
query_vec = vectorizer.transform([query])
similarities = cosine_similarity(query_vec, doc_vectors)[0]

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
colors = ['#C96442' if s == max(similarities) else '#D4A373' for s in similarities]
plt.bar(range(len(similarities)), similarities, color=colors)
plt.xticks(range(len(similarities)), [f'Doc {i}' for i in range(len(documents))])
plt.ylabel('Cosine Similarity')
plt.title(f'Query: "{query}" vs Documents')
plt.tight_layout()
plt.savefig('rag_retrieval_demo.png', dpi=100)
plt.show()
print(f"\n最相关文档: Doc {np.argmax(similarities)} (相似度: {max(similarities):.4f})")
print(f"内容: {documents[np.argmax(similarities)]}")

## Step 3: 检索 + 增强生成（Retrieval + Augmented Generation）

完整 RAG 流程：
1. 用户提问 -> 向量化 query
2. 检索最相关的 top-k 文档块
3. 把检索结果拼接到 prompt 中（增强）
4. 将增强后的 prompt 送入 LLM 生成回答

In [ ]:
def rag_retrieve(query, doc_vectors, documents, vectorizer, top_k=2):
    """检索与 query 最相关的 top_k 文档块"""
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, doc_vectors)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]
    results = [(documents[i], sims[i]) for i in top_indices]
    return results

def rag_build_prompt(query, retrieved_docs):
    """构建增强后的 prompt"""
    context = "\n\n".join([f"[参考资料 {i+1}] {doc}" for i, (doc, _) in enumerate(retrieved_docs)])
    prompt = f"""请根据以下参考资料回答问题。如果资料中没有相关信息，请说明。

---参考资料---
{context}
---

问题：{query}
回答："""
    return prompt

# 演示完整流程
queries = [
    "什么是 RAG？",
    "Transformer 是什么时候提出的？",
    "向量数据库有什么用？"
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Question: {q}")
    results = rag_retrieve(q, doc_vectors, documents, vectorizer, top_k=2)
    print(f"Retrieved:")
    for doc, score in results:
        print(f"   [{score:.4f}] {doc[:60]}...")
    prompt = rag_build_prompt(q, results)
    print(f"\nGenerated Prompt ({len(prompt)} chars):")
    print(prompt[:300] + "...")

## 总结

### RAG 的核心价值
- **无需重训练**：新知识只需更新向量数据库，不用微调模型
- **可追溯**：回答基于具体文档，可溯源验证
- **减少幻觉**：有参考资料约束，模型编造的概率大幅降低

### RAG vs 微调

| 维度 | RAG | 微调 |
|------|-----|------|
| 知识更新 | 实时（更新数据库） | 需要重新训练 |
| 成本 | 较低（检索开销） | 较高（GPU 训练） |
| 适用场景 | 事实性问答、知识库查询 | 风格适应、特定任务优化 |
| 幻觉控制 | 较好 | 一般 |

### 生产级 RAG 的关键工程
- **Embedding 模型**：Sentence-BERT、BGE、OpenAI text-embedding
- **向量数据库**：FAISS（本地）、Milvus（分布式）、Pinecone（云服务）
- **Chunking 策略**：固定长度 / 递归 / 语义切分
- **Reranking**：检索后用 Cross-Encoder 精排

## 课后思考
1. 如果知识库很大（百万级文档），检索速度怎么保证？
2. RAG 和微调能不能同时用？什么场景需要？
3. 如果检索到了错误的文档，RAG 反而会误导模型——如何缓解？